In [1]:

!pip -q install kagglehub

import kagglehub

path = kagglehub.dataset_download(
    "wordsforthewise/lending-club"
)

print("Dataset downloaded to:", path)


Using Colab cache for faster access to the 'lending-club' dataset.
Dataset downloaded to: /kaggle/input/lending-club


In [2]:

import os

print(os.listdir(path))


['rejected_2007_to_2018Q4.csv.gz', 'accepted_2007_to_2018Q4.csv.gz', 'accepted_2007_to_2018q4.csv', 'rejected_2007_to_2018q4.csv']


In [3]:

import pandas as pd
import os

file_path = os.path.join(
    path,
    "accepted_2007_to_2018Q4.csv.gz"
)

df = pd.read_csv(
    file_path,
    compression="gzip",
    nrows=10000,
    low_memory=False
)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

display(df.head())


Rows: 10000
Columns: 151


,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,NaN,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,66310712,NaN,35000.0,35000.0,35000.0,60 months,14.85,829.90,C,C5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,68476807,NaN,10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:

import pandas as pd

total_rows = 0

for chunk in pd.read_csv(
    file_path,
    compression="gzip",
    chunksize=100000,
    usecols=["loan_status"]
):
    total_rows += len(chunk)

print("Total rows:", total_rows)
print("Total columns:", df.shape[1])



Total rows: 2260701
Total columns: 151


In [5]:

status_counts = {}

for chunk in pd.read_csv(
    file_path,
    compression="gzip",
    usecols=["loan_status"],
    chunksize=100000
):
    counts = chunk["loan_status"].value_counts(
        dropna=False
    )

    for status, count in counts.items():
        status_counts[status] = (
            status_counts.get(status, 0) + count
        )

status_df = pd.DataFrame(
    list(status_counts.items()),
    columns=["Loan Status", "Count"]
)

status_df = status_df.sort_values(
    by="Count",
    ascending=False
)

display(status_df)


,Loan Status,Count
0,Fully Paid,1076751
2,Current,878317
1,Charged Off,268559
3,Late (31-120 days),21467
4,In Grace Period,8436
5,Late (16-30 days),4349
8,Does not meet the credit policy. Status:Fully ...,1988
9,Does not meet the credit policy. Status:Charge...,761
6,Default,40
7,NaN,33


In [6]:

# Count the final loan outcomes
fully_paid = status_counts.get("Fully Paid", 0)
charged_off = status_counts.get("Charged Off", 0)
default = status_counts.get("Default", 0)

non_default = fully_paid
default_risk = charged_off + default
total_final = non_default + default_risk

print("Non-default (0):", non_default)
print("Default (1):", default_risk)
print("Total selected rows:", total_final)

print(
    "Default percentage:",
    round(default_risk / total_final * 100, 2),
    "%"
)

print(
    "Non-default percentage:",
    round(non_default / total_final * 100, 2),
    "%"
)


Non-default (0): 1076751
Default (1): 268599
Total selected rows: 1345350
Default percentage: 19.96 %
Non-default percentage: 80.04 %


In [8]:

# Check whether important columns exist in the dataset

candidate_features = [
    "loan_amnt",
    "term",
    "int_rate",
    "installment",
    "grade",
    "sub_grade",
    "emp_length",
    "home_ownership",
    "annual_inc",
    "verification_status",
    "purpose",
    "addr_state",
    "dti",
    "delinq_2yrs",
    "earliest_cr_line",
    "fico_range_low",
    "fico_range_high",
    "inq_last_6mths",
    "open_acc",
    "pub_rec",
    "revol_bal",
    "revol_util",
    "total_acc",
    "mort_acc",
    "pub_rec_bankruptcies"
]

feature_check = pd.DataFrame({
    "Feature": candidate_features,
    "Exists in Dataset": [
        col in df.columns
        for col in candidate_features
    ]
})

display(feature_check)

print(
    "Number of available candidate features:",
    feature_check["Exists in Dataset"].sum()
)


,Feature,Exists in Dataset
0,loan_amnt,True
1,term,True
2,int_rate,True
3,installment,True
4,grade,True
5,sub_grade,True
6,emp_length,True
7,home_ownership,True
8,annual_inc,True
9,verification_status,True


Number of available candidate features: 25


In [9]:

# Inspect the 25 candidate features in our 10,000-row sample

available_features = [
    col for col in candidate_features
    if col in df.columns
]

feature_details = pd.DataFrame({
    "Feature": available_features,
    "Data Type": [
        str(df[col].dtype)
        for col in available_features
    ],
    "Missing %": [
        round(df[col].isna().mean() * 100, 2)
        for col in available_features
    ],
    "Example Value": [
        str(df[col].dropna().iloc[0])
        if df[col].notna().any()
        else "All missing"
        for col in available_features
    ]
})

display(feature_details)



,Feature,Data Type,Missing %,Example Value
0,loan_amnt,float64,0.00,3600.0
1,term,object,0.00,36 months
2,int_rate,float64,0.00,13.99
3,installment,float64,0.00,123.03
4,grade,object,0.00,C
5,sub_grade,object,0.00,C4
6,emp_length,object,5.49,10+ years
7,home_ownership,object,0.00,MORTGAGE
8,annual_inc,float64,0.00,55000.0
9,verification_status,object,0.00,Not Verified


In [10]:

# Inspect candidate features across all selected final-outcome loans

import pandas as pd

selected_statuses = [
    "Fully Paid",
    "Charged Off",
    "Default"
]

columns_to_read = available_features + ["loan_status"]

missing_counts = pd.Series(
    0, index=available_features, dtype="int64"
)

selected_rows = 0

for chunk in pd.read_csv(
    file_path,
    compression="gzip",
    usecols=columns_to_read,
    chunksize=100000,
    low_memory=False
):
    chunk = chunk[
        chunk["loan_status"].isin(selected_statuses)
    ]

    selected_rows += len(chunk)

    missing_counts += (
        chunk[available_features].isna().sum()
    )

full_feature_report = pd.DataFrame({
    "Feature": available_features,
    "Missing Count": missing_counts.values,
    "Missing %": (
        missing_counts.values / selected_rows * 100
    ).round(2)
})

full_feature_report = full_feature_report.sort_values(
    "Missing %",
    ascending=False
)

print("Selected final-outcome rows:", selected_rows)

display(full_feature_report)



Selected final-outcome rows: 1345350


,Feature,Missing Count,Missing %
6,emp_length,78516,5.84
23,mort_acc,47281,3.51
21,revol_util,857,0.06
24,pub_rec_bankruptcies,697,0.05
12,dti,374,0.03
0,loan_amnt,0,0.00
1,term,0,0.00
3,installment,0,0.00
2,int_rate,0,0.00
8,annual_inc,0,0.00
